In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import concordance_index_censored
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [8]:
import os
os.chdir('/Users/parthshringarpure/Desktop/AI/Projects/luad_survival')

# Load our DIY CIBERSORT immune features (478 patients × 22 cell types)
immune = pd.read_csv('data/processed/immune_features_cibersort.csv', index_col=0)

# Load clinical survival labels
clinical = pd.read_csv('data/processed/clinical_survival.csv', index_col=0)

# Align patients
common_patients = immune.index.intersection(clinical.index)
immune = immune.loc[common_patients]
clinical = clinical.loc[common_patients]

print(f"Immune features: {immune.shape}")
print(f"Clinical data:   {clinical.shape}")
print(f"Patients aligned: {len(common_patients)}")
print(f"\nCell types:")
for i, col in enumerate(immune.columns, 1):
    print(f"  {i:2d}. {col}")

Immune features: (478, 22)
Clinical data:   (478, 10)
Patients aligned: 478

Cell types:
   1. B cells naive
   2. B cells memory
   3. Plasma cells
   4. T cells CD8
   5. T cells CD4 naive
   6. T cells CD4 memory resting
   7. T cells CD4 memory activated
   8. T cells follicular helper
   9. T cells regulatory (Tregs)
  10. T cells gamma delta
  11. NK cells resting
  12. NK cells activated
  13. Monocytes
  14. Macrophages M0
  15. Macrophages M1
  16. Macrophages M2
  17. Dendritic cells resting
  18. Dendritic cells activated
  19. Mast cells resting
  20. Mast cells activated
  21. Eosinophils
  22. Neutrophils


In [9]:
# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

print(f"Total patients:  {len(y)}")
print(f"Events (deaths): {y['event'].sum()}  ({y['event'].mean()*100:.1f}%)")
print(f"Censored:        {(~y['event']).sum()}  ({(~y['event']).mean()*100:.1f}%)")
print()

# Scale immune features
# Only 22 features so numerical issues are unlikely
# but we scale for consistency with other models
scaler = StandardScaler()
X = scaler.fit_transform(immune)
X = pd.DataFrame(X, index=immune.index, columns=immune.columns)

print(f"Feature matrix shape: {X.shape}")
print(f"Mean (should be ~0):  {X.values.mean():.6f}")
print(f"Std  (should be ~1):  {X.values.std():.6f}")

Total patients:  478
Events (deaths): 121  (25.3%)
Censored:        357  (74.7%)

Feature matrix shape: (478, 22)
Mean (should be ~0):  0.000000
Std  (should be ~1):  0.977008


In [10]:
from sklearn.model_selection import KFold

# Only 22 features — much smaller than expression or dysregulation
# so we can use a wider alpha range without numerical issues
alphas = np.logspace(-3, 1, 50)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex = []
fold_alphas = []
fold_cells  = []

print("Running 5-fold CV Cox-Lasso on immune features...")
print(f"{'Fold':<6} {'Best Alpha':<12} {'Cells kept':<12} {'C-index':<10}")
print("-" * 44)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    best_alpha = None
    best_cindex = 0

    for a in alphas:
        try:
            cox_a = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[a],
                                            fit_baseline_model=True)
            cox_a.fit(X_train, y_train)
            ci = concordance_index_censored(y_train['event'],
                                             y_train['time'],
                                             cox_a.predict(X_train))[0]
            if ci > best_cindex:
                best_cindex = ci
                best_alpha = a
        except:
            continue

    cox_best = CoxnetSurvivalAnalysis(l1_ratio=1.0, alphas=[best_alpha],
                                       fit_baseline_model=True)
    cox_best.fit(X_train, y_train)

    risk_scores = cox_best.predict(X_test)
    ci_test = concordance_index_censored(y_test['event'],
                                          y_test['time'],
                                          risk_scores)[0]

    n_cells = np.sum(cox_best.coef_[:, 0] != 0)
    fold_cindex.append(ci_test)
    fold_alphas.append(best_alpha)
    fold_cells.append(n_cells)

    print(f"{fold:<6} {best_alpha:<12.4f} {n_cells:<12} {ci_test:.4f}")

print("-" * 44)
print(f"\nMean C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")


Running 5-fold CV Cox-Lasso on immune features...
Fold   Best Alpha   Cells kept   C-index   
--------------------------------------------
1      0.0012       20           0.5804
2      0.0010       20           0.5998
3      0.0037       18           0.5256
4      0.0012       19           0.4994
5      0.0012       20           0.4978
--------------------------------------------

Mean C-index: 0.541 ± 0.042


In [11]:
# Save final model on all data
cox_final = CoxnetSurvivalAnalysis(l1_ratio=1.0,
                                    alphas=[np.mean(fold_alphas)],
                                    fit_baseline_model=True)
cox_final.fit(X, y)

with open('models/cox_lasso_immune.pkl', 'wb') as f:
    pickle.dump(cox_final, f)

with open('models/scaler_immune.pkl', 'wb') as f:
    pickle.dump(scaler, f)

results = {
    "model": "Cox-Lasso Immune (DIY CIBERSORT)",
    "n_patients": 478,
    "n_cell_types": 22,
    "cv_cindex_mean": round(float(np.mean(fold_cindex)), 3),
    "cv_cindex_std": round(float(np.std(fold_cindex)), 3),
    "fold_cindices": [round(float(c), 4) for c in fold_cindex],
    "fold_alphas": [round(float(a), 4) for a in fold_alphas],
    "fold_cells_kept": [int(c) for c in fold_cells]
}

with open('outputs/results/cox_lasso_immune_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved: models/cox_lasso_immune.pkl")
print("Saved: models/scaler_immune.pkl")
print("Saved: outputs/results/cox_lasso_immune_summary.json")
print(f"\nFinal result: C-index = {results['cv_cindex_mean']} ± {results['cv_cindex_std']}")

Saved: models/cox_lasso_immune.pkl
Saved: models/scaler_immune.pkl
Saved: outputs/results/cox_lasso_immune_summary.json

Final result: C-index = 0.541 ± 0.042
